# 🎵 Audio Stem Studio (GPU Tezlashtirilgan)

Ushbu Google Colab daftarchasi yordamida **BS-RoFormer** va **Demucs** modellarini **bepul NVIDIA GPU (T4)** da ishga tushirishingiz mumkin.

> **Muhim:** Yuqoridagi menyudan **Runtime -> Change runtime type** bo'limiga kiring va **T4 GPU** tanlanganligiga ishonch hosil qiling.

In [ ]:
# 1. GPU tekshirish
!nvidia-smi

import shutil
if shutil.which("nvidia-smi") is None:
    raise RuntimeError("NVIDIA GPU topilmadi. Runtime -> Change runtime type -> T4 GPU ni tanlang.")

In [ ]:
# 2. Loyihani yuklab olish va CUDA PyTorch bilan paketlarni o'rnatish
!if [ -d /content/stem-separator/.git ]; then git -C /content/stem-separator pull --ff-only; else git clone https://github.com/khan0200/stem-separator.git /content/stem-separator; fi
%cd /content/stem-separator
!pip install -q --upgrade --no-cache-dir --index-url https://download.pytorch.org/whl/cu121 torch torchaudio
!pip install -q --upgrade audio-separator[gpu] demucs fastapi uvicorn python-multipart jinja2 soundfile
!apt-get -y install ffmpeg

# T4 uchun Python CUDA holatini qat'iy tekshirish
import torch
if not torch.cuda.is_available():
    raise RuntimeError(
        f"PyTorch CUDA ishlamayapti: torch={torch.__version__}, "
        f"cuda={torch.version.cuda}. CUDA PyTorch o'rnatilmadi."
    )
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.version.cuda}")
print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# 3. Serverni ishga tushirish va Internetga ochish (Cloudflare Tunnel)
import subprocess
import time
import urllib.request

# Cloudflared o'rnatish
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb

# Web serverni fonda ishga tushirish
server = subprocess.Popen(["python3", "run.py"])
for _ in range(30):
    if server.poll() is not None:
        raise RuntimeError(f"Server ishga tushmadi (exit code: {server.returncode}).")
    try:
        with urllib.request.urlopen("http://127.0.0.1:8000/", timeout=1) as response:
            if response.status == 200:
                break
    except Exception:
        time.sleep(1)
else:
    server.terminate()
    raise RuntimeError("Server 30 soniyada tayyor bo'lmadi.")
print("\n" + "="*60)
print("  🚀 Audio Stem Studio tayyor!")
print("  Quyidagi chiqadigan 'trycloudflare.com' havolasini bosing:")
print("="*60 + "\n")

# Public HTTPS tunnel ochish
!cloudflared tunnel --url http://127.0.0.1:8000